# NER — Named Entity Recognition

Runs NER across the vault using the configured backend and compares results
against the ground truth in `data/ground_truth.yaml`.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, '../src')

from glob import glob
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

from notemine.config import load_config
from notemine.frontmatter_io import load_note
from notemine.ground_truth import load_ground_truth
from notemine.tasks import run_ner

load_dotenv('../.env')
config = load_config('../config.toml')
gt = load_ground_truth('../' + config['vault']['ground_truth'].lstrip('./'))

vault = '../' + config['vault']['path'].lstrip('./')
config['vault']['prompts_dir'] = '../' + config['vault']['prompts_dir'].lstrip('./')

notes = sorted(
    p for p in glob(f'{vault}/**/*.md', recursive=True)
    if not Path(p).name.startswith('index_')
    and not Path(p).suffix == '.yaml'
)
print(f'{len(notes)} notes found')

20 notes found


## Run NER

In [2]:
import time

backend = config['run']['backend']

results = []
for path in notes:
    name = Path(path).name
    post = load_note(path)
    if len(post.content.strip()) < 50:
        continue

    gt_entry = gt.get(name, {})
    gt_entities = [(e['entity'], e['type']) for e in gt_entry.get('entities', [])]

    row = {'file': name, 'lang': gt_entry.get('lang', '?'), 'ground_truth': gt_entities}

    t0 = time.perf_counter()
    try:
        row[backend] = [(e[0], e[1]) for e in run_ner(backend, post.content, config)]
    except Exception as e:
        row[backend] = f'ERROR: {e}'
    row['elapsed_s'] = time.perf_counter() - t0

    results.append(row)
    print(f'  {name}  ({row["elapsed_s"]:.1f}s)')

print(f'\nDone — {len(results)} notes processed')

  long_ai_gezondheidszorg_oncologie_onderzoek_nl.md  (19.1s)
  long_ai_healthcare_oncology_research_en.md  (18.3s)
  long_interview_climate_scientist_sea_level_en.md  (17.0s)
  long_interview_klimaatwetenschapper_zeespiegel_nl.md  (21.0s)
  long_opinie_eu_ai_act_digitaal_beleid_nl.md  (22.5s)
  long_opinion_eu_ai_act_digital_policy_en.md  (16.8s)
  medium_climate_blog_paris_agreement_en.md  (18.9s)
  medium_geschiedenis_gouden_eeuw_nl.md  (18.0s)
  medium_history_dutch_golden_age_en.md  (27.5s)
  medium_klimaat_blog_parijs_akkoord_nl.md  (25.5s)
  medium_reisgids_amsterdam_nl.md  (11.7s)
  medium_travel_guide_amsterdam_en.md  (16.2s)
  short_recensie_samsung_galaxy_s25_ultra_nl.md  (6.7s)
  short_recept_stamppot_nl.md  (7.1s)
  short_recipe_dutch_stamppot_en.md  (4.5s)
  short_review_samsung_galaxy_s25_ultra_en.md  (6.2s)
  short_sport_nieuws_ajax_champions_league_nl.md  (8.1s)
  short_sports_news_ajax_champions_league_en.md  (7.6s)
  short_tech_news_openai_announcement_en.md  (6.7s)
 

In [3]:
import json

out_dir = Path('../results/ner')
out_dir.mkdir(parents=True, exist_ok=True)

for r in results:
    stem = Path(r['file']).stem
    payload = r[backend]
    data = {
        'file': r['file'],
        'backend': backend,
        'model': config[backend]['model'],
        'elapsed_s': round(r['elapsed_s'], 3),
        'ground_truth': r['ground_truth'],
        'entities': payload if not isinstance(payload, str) else [],
        'error': payload if isinstance(payload, str) else None,
    }
    (out_dir / f'{stem}.json').write_text(json.dumps(data, ensure_ascii=False, indent=2))

print(f'Saved {len(results)} files to {out_dir}')

Saved 20 files to ../results/ner


## Summary comparison

Shows how many ground-truth entities each backend matched, plus how many extra it added.

In [4]:
def ner_stats(gt_ents, pred):
    if isinstance(pred, str):
        return pred
    gt_set = set(gt_ents)
    pred_set = set(pred)
    matched = len(gt_set & pred_set)
    missed = len(gt_set - pred_set)
    extra = len(pred_set - gt_set)
    pct = int(100 * matched / len(gt_set)) if gt_set else 0
    return f'{matched}/{len(gt_set)} ({pct}%)  +{extra} extra'

rows = []
for r in results:
    rows.append({
        'file': r['file'],
        'lang': r['lang'],
        'gt_count': len(r['ground_truth']),
        backend: ner_stats(r['ground_truth'], r[backend]),
    })

pd.set_option('display.max_colwidth', 300)
pd.DataFrame(rows)

,file,lang,gt_count,ollama
0,long_ai_gezondheidszorg_oncologie_onderzoek_nl.md,nl,48,11/48 (22%) +10 extra
1,long_ai_healthcare_oncology_research_en.md,en,48,14/48 (29%) +12 extra
2,long_interview_climate_scientist_sea_level_en.md,en,45,7/45 (15%) +20 extra
3,long_interview_klimaatwetenschapper_zeespiegel_nl.md,nl,45,10/45 (22%) +23 extra
4,long_opinie_eu_ai_act_digitaal_beleid_nl.md,nl,47,16/47 (34%) +23 extra
5,long_opinion_eu_ai_act_digital_policy_en.md,en,47,23/47 (48%) +12 extra
6,medium_climate_blog_paris_agreement_en.md,en,41,19/41 (46%) +18 extra
7,medium_geschiedenis_gouden_eeuw_nl.md,nl,36,6/36 (16%) +23 extra
8,medium_history_dutch_golden_age_en.md,en,36,19/36 (52%) +32 extra
9,medium_klimaat_blog_parijs_akkoord_nl.md,nl,41,24/41 (58%) +24 extra


## Inspect a single note

Change `INSPECT` to any filename to see the full entity lists side by side.

In [5]:
INSPECT = Path(notes[1]).name  # change to any note filename

row = next((r for r in results if r['file'] == INSPECT), None)
if row is None:
    print(f'{INSPECT} not found in results')
else:
    gt_set = set(row['ground_truth'])
    pred = set(row[backend]) if not isinstance(row[backend], str) else set()

    all_ents = sorted(gt_set | pred)
    rows = []
    for ent in all_ents:
        rows.append({
            'entity': ent[0],
            'type': ent[1],
            'ground_truth': '✓' if ent in gt_set else '',
            backend: '✓' if ent in pred else '',
        })

    print(f'=== {INSPECT} ===')
    pd.set_option('display.max_colwidth', 50)
    display(pd.DataFrame(rows))

=== long_ai_healthcare_oncology_research_en.md ===


,entity,type,ground_truth,ollama
0,2022,DATE,,✓
1,2024,DATE,,✓
2,2027,DATE,✓,
3,2028,DATE,,✓
4,2030,DATE,,✓
5,Academic Medical Center (AMC),ORG,✓,✓
6,Amsterdam,LOC,✓,
7,Amsterdam UMC,ORG,✓,✓
8,Autoriteit Persoonsgegevens,ORG,✓,✓
9,Boston,LOC,✓,


## (Optional) Save results to frontmatter

Uncomment and run to write results to note files under `notemine.ner.<backend>`.
Run `python main.py reset` to undo.

In [6]:
# from notemine.frontmatter_io import save_note
#
# for path in notes:
#     name = Path(path).name
#     row = next((r for r in results if r['file'] == name), None)
#     if row is None:
#         continue
#     post = load_note(path)
#     notemine = post.metadata.get('notemine', {})
#     ner_data = notemine.get('ner', {})
#     if not isinstance(row.get(backend), str):
#         ner_data[backend] = [list(e) for e in row[backend]]
#     notemine['ner'] = ner_data
#     post.metadata['notemine'] = notemine
#     save_note(path, post)
# print('Saved.')